# Ancestro común más bajo

En un árbol con raíz, el ancestro común más bajo (LCA) de $u$ y $v$ es el nodo más profundo que es antecesor de ambos (contando a cada nodo como antecesor de sí mismo). La versión ingenua iguala profundidades y sube a la vez, en $O(h)$ por consulta. Con *binary lifting* se precalcula el antecesor $2^k$ de cada nodo,

$$\text{arriba}_k(v) = \text{arriba}_{k-1}\big(\text{arriba}_{k-1}(v)\big)$$

en $O(n \log n)$, y cada consulta salta por potencias de dos en $O(\log n)$. Con el LCA se calculan distancias ($d(u, v) = p_u + p_v - 2\,p_{\text{lca}}$), el $k$-ésimo antecesor, caminos y parentescos.

## 1. Un árbol de carpetas

Construir el árbol de un disco a partir de una lista de rutas: el padre de cada carpeta es su ruta sin el último componente y la raíz es `/`. Calcular la profundidad de cada carpeta con un BFS y dibujar el árbol sangrado.

In [1]:
import random
from collections import deque

import networkx as nx

rutas = [
    "/home/ana/fotos/2023", "/home/ana/fotos/2024/verano", "/home/ana/fotos/2024/carnaval",
    "/home/ana/docs/facturas", "/home/ana/docs/tesis/capitulos", "/home/luis/musica/timple",
    "/home/luis/proyectos/web", "/etc/ssh", "/var/log/nginx", "/var/www",
]


def arbol_de_rutas(rutas):
    padre = {"/": "/"}
    for ruta in rutas:
        partes = ruta.strip("/").split("/")
        for k in range(1, len(partes) + 1):
            nodo = "/" + "/".join(partes[:k])
            padre[nodo] = "/" + "/".join(partes[:k - 1]) if k > 1 else "/"
    return padre


def hijos_de(padre):
    hijos = {v: [] for v in padre}
    for v, p in padre.items():
        if v != p:
            hijos[p].append(v)
    return {v: sorted(h) for v, h in hijos.items()}


def profundidades(padre, raiz):
    hijos = hijos_de(padre)
    prof = {raiz: 0}
    cola = deque([raiz])
    while cola:
        u = cola.popleft()
        for h in hijos[u]:
            prof[h] = prof[u] + 1
            cola.append(h)
    return prof


padre = arbol_de_rutas(rutas)
prof = profundidades(padre, "/")
hijos = hijos_de(padre)


def dibuja(nodo, nivel=0):
    print("  " * nivel + (nodo.rsplit("/", 1)[-1] or "/"))
    for h in hijos[nodo]:
        dibuja(h, nivel + 1)


dibuja("/")
print(len(padre), "carpetas; profundidad máxima", max(prof.values()))
assert all(prof[v] == v.count("/") for v in padre if v != "/")

/
  etc
    ssh
  home
    ana
      docs
        facturas
        tesis
          capitulos
      fotos
        2023
        2024
          carnaval
          verano
    luis
      musica
        timple
      proyectos
        web
  var
    log
      nginx
    www
23 carpetas; profundidad máxima 5


## 2. LCA ingenuo

Escribir `lca_ingenuo(u, v)`: subir el nodo más profundo hasta igualar profundidades y después subir los dos a la vez hasta que coincidan. Comprobarlo contra la definición: el primer antecesor de $v$ que está en el conjunto de antecesores de $u$. Usarlo para encontrar la carpeta común más profunda de dos ficheros.

In [2]:
def lca_ingenuo(u, v, padre, prof):
    while prof[u] > prof[v]:
        u = padre[u]
    while prof[v] > prof[u]:
        v = padre[v]
    while u != v:
        u, v = padre[u], padre[v]
    return u


def antecesores(v, padre):
    camino = [v]
    while padre[camino[-1]] != camino[-1]:
        camino.append(padre[camino[-1]])
    return camino


def lca_por_definicion(u, v, padre):
    de_u = set(antecesores(u, padre))
    return next(a for a in antecesores(v, padre) if a in de_u)


for u, v in [("/home/ana/fotos/2024/verano", "/home/ana/fotos/2023"),
             ("/home/ana/docs/tesis/capitulos", "/home/luis/musica/timple"),
             ("/var/www", "/etc/ssh"),
             ("/home/ana/docs", "/home/ana/docs/facturas")]:
    print(f"{u}  y  {v}  →  {lca_ingenuo(u, v, padre, prof)}")

for u in padre:
    for v in padre:
        assert lca_ingenuo(u, v, padre, prof) == lca_por_definicion(u, v, padre)

/home/ana/fotos/2024/verano  y  /home/ana/fotos/2023  →  /home/ana/fotos
/home/ana/docs/tesis/capitulos  y  /home/luis/musica/timple  →  /home
/var/www  y  /etc/ssh  →  /
/home/ana/docs  y  /home/ana/docs/facturas  →  /home/ana/docs


## 3. Árboles aleatorios para las pruebas

Escribir un generador de árboles aleatorios de $n$ nodos (`0` es la raíz y el padre de cada nodo es un nodo anterior) con un parámetro que los haga más "alargados", para tener casos con altura cercana a $n$, que son los que penalizan al método ingenuo. Comprobar el LCA ingenuo en ellos contra `nx.lowest_common_ancestor`.

In [3]:
def arbol_aleatorio(n, alargado, rnd):
    padre = {0: 0}
    for v in range(1, n):
        padre[v] = v - 1 if rnd.random() < alargado else rnd.randrange(v)
    return padre


rnd = random.Random(6)
for alargado in (0.0, 0.5, 0.95):
    p = arbol_aleatorio(2000, alargado, rnd)
    print(f"alargado = {alargado:<4}: altura {max(profundidades(p, 0).values())}")

for _ in range(30):
    p = arbol_aleatorio(rnd.randint(1, 60), rnd.random(), rnd)
    pr = profundidades(p, 0)
    g = nx.DiGraph((a, b) for b, a in p.items() if a != b)
    g.add_node(0)
    for _ in range(50):
        u, v = rnd.randrange(len(p)), rnd.randrange(len(p))
        assert lca_ingenuo(u, v, p, pr) == nx.lowest_common_ancestor(g, u, v)

alargado = 0.0 : altura 14
alargado = 0.5 : altura 28
alargado = 0.95: altura 268


## 4. Binary lifting y k-ésimo antecesor

Precalcular la tabla `arriba[k][v]` (antecesor $2^k$ de $v$, con la raíz como su propio padre) y escribir `k_esimo(v, k)`, que descompone $k$ en bits y da un salto por cada bit. Comprobarlo contra subir $k$ veces de uno en uno y mostrar, por ejemplo, la carpeta dos niveles por encima de un fichero.

In [4]:
class Ascensor:
    def __init__(self, padre, raiz):
        self.prof = profundidades(padre, raiz)
        self.arriba = [dict(padre)]
        for _ in range(max(self.prof.values()).bit_length()):
            anterior = self.arriba[-1]
            self.arriba.append({v: anterior[anterior[v]] for v in padre})

    def k_esimo(self, v, k):
        """Antecesor k niveles por encima de v, o None si no existe."""
        if k > self.prof[v]:
            return None
        bit = 0
        while k:
            if k & 1:
                v = self.arriba[bit][v]
            k >>= 1
            bit += 1
        return v

    def lca(self, u, v):
        if self.prof[u] < self.prof[v]:
            u, v = v, u
        u = self.k_esimo(u, self.prof[u] - self.prof[v])
        if u == v:
            return u
        for nivel in reversed(self.arriba):
            if nivel[u] != nivel[v]:
                u, v = nivel[u], nivel[v]
        return self.arriba[0][u]

    def distancia(self, u, v):
        return self.prof[u] + self.prof[v] - 2 * self.prof[self.lca(u, v)]


carpetas = Ascensor(padre, "/")
print("Niveles de la tabla:", len(carpetas.arriba))
for k in range(5):
    print(k, carpetas.k_esimo("/home/ana/fotos/2024/verano", k))

for _ in range(30):
    p = arbol_aleatorio(rnd.randint(1, 300), rnd.random(), rnd)
    asc = Ascensor(p, 0)
    for _ in range(100):
        v = rnd.randrange(len(p))
        k = rnd.randint(0, asc.prof[v] + 2)
        esperado = antecesores(v, p)[k] if k <= asc.prof[v] else None
        assert asc.k_esimo(v, k) == esperado

Niveles de la tabla: 4
0 /home/ana/fotos/2024/verano
1 /home/ana/fotos/2024
2 /home/ana/fotos
3 /home/ana
4 /home


## 5. LCA con binary lifting

Con la tabla anterior, el LCA iguala profundidades con `k_esimo` y luego, de la potencia mayor a la menor, sube ambos nodos siempre que sus antecesores sigan siendo distintos; al final el LCA es el padre. Comprobarlo contra el método ingenuo en árboles aleatorios y contar los pasos de cada uno en un árbol muy alargado.

In [5]:
def pasos_ingenuo(u, v, padre, prof):
    pasos = 0
    while u != v:
        if prof[u] >= prof[v]:
            u = padre[u]
        else:
            v = padre[v]
        pasos += 1
    return pasos


for _ in range(30):
    p = arbol_aleatorio(rnd.randint(1, 300), rnd.random(), rnd)
    asc = Ascensor(p, 0)
    for _ in range(200):
        u, v = rnd.randrange(len(p)), rnd.randrange(len(p))
        assert asc.lca(u, v) == lca_ingenuo(u, v, p, asc.prof)

p = arbol_aleatorio(20_000, 0.999, rnd)
asc = Ascensor(p, 0)
u, v = max(p, key=asc.prof.get), 5
print(f"Profundidades {asc.prof[u]} y {asc.prof[v]}: LCA {asc.lca(u, v)}")
print(f"Pasos del ingenuo: {pasos_ingenuo(u, v, p, asc.prof)}; saltos máximos del binary lifting: {2 * len(asc.arriba)}")

Profundidades 7041 y 5: LCA 5
Pasos del ingenuo: 7036; saltos máximos del binary lifting: 28


## 6. Distancia y ruta relativa entre carpetas

La distancia entre dos nodos es $p_u + p_v - 2\,p_{\text{lca}}$. Usarla para calcular cuántos `cd` hacen falta para ir de una carpeta a otra y escribir la ruta relativa (`../../docs/tesis`). Comprobar la distancia con un BFS en el árbol no dirigido y la ruta con `posixpath.relpath`.

In [6]:
import posixpath


def ruta_relativa(origen, destino, asc):
    comun = asc.lca(origen, destino)
    subir = [".."] * (asc.prof[origen] - asc.prof[comun])
    bajar = antecesores(destino, asc.arriba[0])[:asc.prof[destino] - asc.prof[comun]]
    return "/".join(subir + [d.rsplit("/", 1)[-1] for d in reversed(bajar)]) or "."


def distancia_bfs(u, v, padre):
    g = nx.Graph((a, b) for a, b in padre.items() if a != b)
    return nx.shortest_path_length(g, u, v)


for origen, destino in [("/home/ana/fotos/2024/verano", "/home/ana/docs/tesis"),
                        ("/var/log/nginx", "/var/www"),
                        ("/home/luis", "/home/luis/proyectos/web"),
                        ("/etc/ssh", "/etc/ssh")]:
    rel = ruta_relativa(origen, destino, carpetas)
    print(f"{origen} → {destino}: {carpetas.distancia(origen, destino)} saltos, cd {rel}")
    assert rel == posixpath.relpath(destino, origen)

nodos = list(padre)
for u in nodos:
    for v in nodos:
        if u != v:
            assert carpetas.distancia(u, v) == distancia_bfs(u, v, padre)
        assert ruta_relativa(u, v, carpetas) == posixpath.relpath(v, u)

/home/ana/fotos/2024/verano → /home/ana/docs/tesis: 5 saltos, cd ../../../docs/tesis
/var/log/nginx → /var/www: 3 saltos, cd ../../www
/home/luis → /home/luis/proyectos/web: 2 saltos, cd proyectos/web
/etc/ssh → /etc/ssh: 0 saltos, cd .


## 7. El nodo k-ésimo del camino

Escribir `nodo_en_camino(u, v, k)`, que devuelve el nodo que está a $k$ pasos de $u$ en el camino hacia $v$: si $k$ no pasa del tramo de subida hasta el LCA es el $k$-ésimo antecesor de $u$, y si no, un antecesor de $v$. Comprobarlo contra el camino de `nx.shortest_path`.

In [7]:
def nodo_en_camino(u, v, k, asc):
    comun = asc.lca(u, v)
    subida = asc.prof[u] - asc.prof[comun]
    total = subida + asc.prof[v] - asc.prof[comun]
    if not 0 <= k <= total:
        return None
    return asc.k_esimo(u, k) if k <= subida else asc.k_esimo(v, total - k)


u, v = "/home/ana/fotos/2024/carnaval", "/home/luis/musica/timple"
print([nodo_en_camino(u, v, k, carpetas) for k in range(carpetas.distancia(u, v) + 1)])

for _ in range(20):
    p = arbol_aleatorio(rnd.randint(1, 80), rnd.random(), rnd)
    asc = Ascensor(p, 0)
    g = nx.Graph((a, b) for a, b in p.items() if a != b)
    g.add_node(0)
    for _ in range(50):
        u, v = rnd.randrange(len(p)), rnd.randrange(len(p))
        camino = nx.shortest_path(g, u, v)
        assert [nodo_en_camino(u, v, k, asc) for k in range(len(camino) + 1)] == camino + [None]

['/home/ana/fotos/2024/carnaval', '/home/ana/fotos/2024', '/home/ana/fotos', '/home/ana', '/home', '/home/luis', '/home/luis/musica', '/home/luis/musica/timple']


## 8. Parentescos en un árbol genealógico

En un árbol genealógico, el parentesco entre dos personas depende de cuántas generaciones las separan de su antepasado común más cercano. Escribir `parentesco(a, b)` que use el LCA para devolver "padre/madre", "abuelo/a", "hermano/a", "tío/a", "sobrino/a", "primo/a", "primo/a segundo/a"… y comprobarlo con varios pares de una familia inventada.

In [8]:
familia = {  # hijo: progenitor (una sola línea por simplicidad)
    "Bentejuí": "Bentejuí",
    "Guacimara": "Bentejuí", "Tinguaro": "Bentejuí",
    "Dácil": "Guacimara", "Yeray": "Guacimara", "Acaymo": "Tinguaro",
    "Nayra": "Dácil", "Itahisa": "Yeray", "Ancor": "Acaymo",
    "Aday": "Nayra", "Chaxiraxi": "Ancor",
}
genealogia = Ascensor(familia, "Bentejuí")
ASCENDIENTES = {1: "padre/madre", 2: "abuelo/a", 3: "bisabuelo/a", 4: "tatarabuelo/a"}
DESCENDIENTES = {1: "hijo/a", 2: "nieto/a", 3: "bisnieto/a", 4: "tataranieto/a"}
ORDINALES = {2: "segundo/a", 3: "tercero/a", 4: "cuarto/a"}


def parentesco(a, b, asc=genealogia):
    """Qué es a respecto de b."""
    comun = asc.lca(a, b)
    da, db = asc.prof[a] - asc.prof[comun], asc.prof[b] - asc.prof[comun]
    if da == db == 0:
        return "la misma persona"
    if da == 0:
        return ASCENDIENTES[db]
    if db == 0:
        return DESCENDIENTES[da]
    if min(da, db) == 1:
        if da == db:
            return "hermano/a"
        lejania = abs(da - db)
        # tío abuelo, sobrino nieto…: el resto del nombre sale de las generaciones de más
        if da < db:
            return "tío/a" + ("" if lejania == 1 else " " + ASCENDIENTES[lejania])
        return "sobrino/a" + ("" if lejania == 1 else " " + DESCENDIENTES[lejania])
    grado = min(da, db) - 1
    texto = "primo/a" + (" " + ORDINALES[grado] if grado > 1 else "")
    diferencia = abs(da - db)
    return texto + (f", con {diferencia} generación{'es' if diferencia > 1 else ''} de diferencia" if diferencia else "")


pares = [
    ("Guacimara", "Dácil", "padre/madre"),
    ("Bentejuí", "Aday", "tatarabuelo/a"),
    ("Itahisa", "Yeray", "hijo/a"),
    ("Dácil", "Yeray", "hermano/a"),
    ("Tinguaro", "Nayra", "tío/a abuelo/a"),
    ("Aday", "Yeray", "sobrino/a nieto/a"),
    ("Dácil", "Acaymo", "primo/a"),
    ("Nayra", "Itahisa", "primo/a"),
    ("Nayra", "Ancor", "primo/a segundo/a"),
    ("Aday", "Chaxiraxi", "primo/a tercero/a"),
    ("Aday", "Ancor", "primo/a segundo/a, con 1 generación de diferencia"),
]
for a, b, esperado in pares:
    print(f"{a} es {parentesco(a, b)} de {b}")
    assert parentesco(a, b) == esperado

Guacimara es padre/madre de Dácil
Bentejuí es tatarabuelo/a de Aday
Itahisa es hijo/a de Yeray
Dácil es hermano/a de Yeray
Tinguaro es tío/a abuelo/a de Nayra
Aday es sobrino/a nieto/a de Yeray
Dácil es primo/a de Acaymo
Nayra es primo/a de Itahisa
Nayra es primo/a segundo/a de Ancor
Aday es primo/a tercero/a de Chaxiraxi
Aday es primo/a segundo/a, con 1 generación de diferencia de Ancor


## 9. Mayor tramo de carretera en un camino

Una red de carreteras sin ciclos (un árbol) une pueblos con tramos de distinta longitud. Para planificar la autonomía de un vehículo eléctrico interesa, para cada viaje, el tramo más largo que se recorre. Ampliar el binary lifting para guardar junto a cada salto $2^k$ el máximo de las aristas que cubre, y responder cada consulta en $O(\log n)$. Comprobarlo contra recorrer el camino de `nx.shortest_path`.

In [9]:
class AscensorConMaximo(Ascensor):
    def __init__(self, padre, raiz, peso):
        super().__init__(padre, raiz)
        self.maximo = [{v: peso.get(v, 0) for v in padre}]  # peso de la arista v-padre(v)
        for k in range(1, len(self.arriba)):
            anterior, salto = self.maximo[-1], self.arriba[k - 1]
            self.maximo.append({v: max(anterior[v], anterior[salto[v]]) for v in padre})

    def sube(self, v, k):
        mayor, bit = 0, 0
        while k:
            if k & 1:
                mayor = max(mayor, self.maximo[bit][v])
                v = self.arriba[bit][v]
            k >>= 1
            bit += 1
        return mayor

    def tramo_maximo(self, u, v):
        comun = self.lca(u, v)
        return max(self.sube(u, self.prof[u] - self.prof[comun]), self.sube(v, self.prof[v] - self.prof[comun]))


carreteras = {  # pueblo: (pueblo más cercano a la capital, km)
    "Valverde": ("Valverde", 0), "Tamaduste": ("Valverde", 6), "La Caleta": ("Valverde", 7),
    "San Andrés": ("Valverde", 10), "Isora": ("San Andrés", 9), "Las Playas": ("Isora", 8),
    "Frontera": ("San Andrés", 18), "Tigaday": ("Frontera", 2), "Sabinosa": ("Tigaday", 9),
    "La Restinga": ("San Andrés", 30), "El Pinar": ("La Restinga", 12),
}
padre_c = {v: p for v, (p, _) in carreteras.items()}
red = AscensorConMaximo(padre_c, "Valverde", {v: km for v, (_, km) in carreteras.items()})
for a, b in [("Sabinosa", "Tamaduste"), ("Las Playas", "Tigaday"), ("El Pinar", "Isora"), ("Tigaday", "Sabinosa")]:
    print(f"{a} → {b}: tramo más largo {red.tramo_maximo(a, b)} km")


def tramo_maximo_bruto(u, v, padre, peso):
    g = nx.Graph()
    g.add_nodes_from(padre)
    g.add_edges_from((a, b, {"km": peso[a]}) for a, b in padre.items() if a != b)
    camino = nx.shortest_path(g, u, v)
    return max((g.edges[a, b]["km"] for a, b in zip(camino, camino[1:])), default=0)


for _ in range(20):
    p = arbol_aleatorio(rnd.randint(1, 100), rnd.random(), rnd)
    peso = {v: rnd.randint(1, 100) for v in p if v != 0}
    asc = AscensorConMaximo(p, 0, peso)
    for _ in range(50):
        u, v = rnd.randrange(len(p)), rnd.randrange(len(p))
        assert asc.tramo_maximo(u, v) == tramo_maximo_bruto(u, v, p, peso)

Sabinosa → Tamaduste: tramo más largo 18 km
Las Playas → Tigaday: tramo más largo 18 km
El Pinar → Isora: tramo más largo 30 km
Tigaday → Sabinosa: tramo más largo 9 km


## 10. LCA en O(1) con el recorrido de Euler

Otra forma de resolver el LCA: recorrer el árbol en profundidad anotando cada nodo cada vez que se pasa por él (recorrido de Euler, $2n - 1$ entradas). El LCA de $u$ y $v$ es el nodo de menor profundidad entre sus primeras apariciones, así que el problema se reduce a mínimos en rangos y, con una sparse table, cada consulta cuesta $O(1)$. Implementarlo de forma iterativa y comprobarlo contra binary lifting.

In [10]:
class LcaEuler:
    def __init__(self, padre, raiz):
        hijos = hijos_de(padre)
        self.prof = profundidades(padre, raiz)
        self.euler, self.primera = [], {}
        pila = [(raiz, iter(hijos[raiz]))]
        self._anota(raiz)
        while pila:
            nodo, pendientes = pila[-1]
            hijo = next(pendientes, None)
            if hijo is None:
                pila.pop()
                if pila:
                    self._anota(pila[-1][0])
            else:
                self._anota(hijo)
                pila.append((hijo, iter(hijos[hijo])))
        clave = lambda x: self.prof[x]
        self.tabla = [self.euler]
        while 1 << len(self.tabla) <= len(self.euler):
            ant, mitad = self.tabla[-1], 1 << (len(self.tabla) - 1)
            self.tabla.append([min(ant[i], ant[i + mitad], key=clave) for i in range(len(ant) - mitad)])

    def _anota(self, nodo):
        self.primera.setdefault(nodo, len(self.euler))
        self.euler.append(nodo)

    def lca(self, u, v):
        l, r = sorted((self.primera[u], self.primera[v]))
        r += 1
        k = (r - l).bit_length() - 1
        return min(self.tabla[k][l], self.tabla[k][r - (1 << k)], key=self.prof.get)


euler = LcaEuler(padre, "/")
print("Recorrido de Euler:", len(euler.euler), "entradas para", len(padre), "carpetas")
print(euler.lca("/home/ana/fotos/2024/verano", "/home/ana/docs/facturas"))
assert len(euler.euler) == 2 * len(padre) - 1

for _ in range(30):
    p = arbol_aleatorio(rnd.randint(1, 400), rnd.random(), rnd)
    e, asc = LcaEuler(p, 0), Ascensor(p, 0)
    for _ in range(200):
        u, v = rnd.randrange(len(p)), rnd.randrange(len(p))
        assert e.lca(u, v) == asc.lca(u, v)

Recorrido de Euler: 45 entradas para 23 carpetas
/home/ana
